## Safe public paste

- **Gradio:** run the cell below to embed the UI in-notebook (`inline=True`).
- **CLI via uv:** from the **agents repo root** (where `pyproject.toml` lives), run:
  `uv run python 2_openai/community_contributions/safe_paste_public/run_demo.py 2_openai/community_contributions/safe_paste_public/sample_inputs/leaky_log.txt`
- Set `OPENAI_API_KEY` (e.g. repo `.env`). The setup cell locates `safe_paste_public` whether your kernel **cwd** is this folder or the **agents repo root**.

In [6]:
from pathlib import Path
import sys


def _find_safe_paste_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "orchestrator.py").is_file() and (d / "app.py").is_file():
            return d
        nested = d / "2_openai" / "community_contributions" / "safe_paste_public"
        if (nested / "orchestrator.py").is_file():
            return nested
    raise RuntimeError(
        "Could not find safe_paste_public (orchestrator.py + app.py)."
    )


ROOT = _find_safe_paste_root()
sys.path.insert(0, str(ROOT))

from env_setup import load_repo_env

load_repo_env()

### Gradio UI (inline in this notebook)

In [ ]:
from app import build_ui

demo = build_ui()
demo.launch(inline=True, share=False, quiet=True)

### Optional: call the orchestrator directly (no Gradio)

In [ ]:
from orchestrator import SafePasteManager

text = (ROOT / "sample_inputs" / "clean_traceback.txt").read_text(encoding="utf-8")


async def show():
    async for chunk in SafePasteManager().run(text):
        print(chunk, end="")


await show()